# Reunión 13 — Factores de riesgo CV e IBERLIFERISK

En este notebook:

1. **Exploración IBERLIFERISK.** De las pacientes con score IBERLIFERISK por encima del corte (>1 y >1.5), se revisa cuántas presentan además otros factores de riesgo cardiovascular (HTA crónica, IMC alto, colesterol alto, TAS alta, tabaco, diabetes…).
2. **Flag de riesgo en `datos.csv`.** Se crea una variable binaria que vale **1** si la paciente tiene **alguno** de los factores de riesgo de Framingham marcado (positivo/alterado) y **0** si no tiene ninguno.

**Variables usadas para el flag** (subconjunto de Framingham, reunión 11):
`colesterol_total`, `hdl`, `ta_sistolica`, `fuma_post`, `diabetes_mellitus_1_2`.

> Se **excluyen** a propósito `tomo_Antihipertensivo` (ya tratado en otro análisis) y `edad_actual` (no se usa como factor aquí).

**Cortes clínicos** (los mismos de la reunión 12):

| Factor | Condición para marcar riesgo |
|---|---|
| Colesterol total alto | `colesterol_total` ≥ 200 mg/dL |
| HDL bajo (mujer) | `hdl` < 50 mg/dL |
| TAS alta | `ta_sistolica` ≥ 140 mmHg |
| Tabaco | `fuma_post` ∈ {`Si`, `Ex_Fumadora`} |
| Diabetes | `diabetes_mellitus_1_2` == 1 |


In [1]:
import pandas as pd
import numpy as np

# datos.csv -> 459 pacientes, contiene las variables de Framingham
df = pd.read_csv('datos.csv')
df['id'] = df['id'].astype(float)

# El score IBERLIFERISK no está en datos.csv; vive en datos35.csv (mismas 459 pacientes)
df_ib = pd.read_csv('datos35.csv')
df_ib['id'] = df_ib['id'].astype(float)
COL_IBER = 'resultado_iberliferisk_score_hasta_los_75_anos'
df_ib['iberliferisk'] = pd.to_numeric(df_ib[COL_IBER], errors='coerce')

print(f'datos.csv: {len(df)} pacientes')
print(f'IBERLIFERISK disponible en {df_ib["iberliferisk"].notna().sum()} pacientes')
print(df_ib['iberliferisk'].describe())

datos.csv: 459 pacientes
IBERLIFERISK disponible en 313 pacientes
count    313.000000
mean       0.563067
std        1.212704
min        0.030000
25%        0.170000
50%        0.320000
75%        0.560000
max       18.560000
Name: iberliferisk, dtype: float64


## 1. Exploración: IBERLIFERISK por encima del corte

De las pacientes con IBERLIFERISK alto, ¿cuántas acumulan además otros factores de riesgo?
Se muestran ambos cortes (**>1** y **>1.5**) para no fijar el umbral a priori.

In [2]:
# Cortes clínicos (reutilizados de reunión 12)
CT_ALTO  = 200   # colesterol total alto (mg/dL)
HDL_MIN  = 50    # HDL bajo en mujer (mg/dL)
TAS_ALTA = 140   # TAS alta (mmHg)
IMC_ALTO = 30    # obesidad (kg/m2) — solo para la exploración

# Unimos el score IBER a las variables clínicas de datos.csv
df_expl = df.merge(df_ib[['id', 'iberliferisk']], on='id', how='left')

def perfil_riesgo(sub):
    """Conteo de factores de riesgo en un subconjunto de pacientes."""
    return pd.Series({
        'N pacientes'        : len(sub),
        'HTA crónica'        : int((sub['hta_cronica'] == 1).sum()),
        'IMC ≥ 30'           : int((sub['imc_actual'] >= IMC_ALTO).sum()),
        'Colesterol ≥ 200'   : int((sub['colesterol_total'] >= CT_ALTO).sum()),
        'HDL < 50'           : int((sub['hdl'] < HDL_MIN).sum()),
        'TAS ≥ 140'          : int((sub['ta_sistolica'] >= TAS_ALTA).sum()),
        'Tabaco (Si/Ex)'     : int(sub['fuma_post'].isin(['Si', 'Ex_Fumadora']).sum()),
        'Diabetes'           : int((sub['diabetes_mellitus_1_2'] == 1).sum()),
    })

resumen = pd.DataFrame({
    'IBER > 1'   : perfil_riesgo(df_expl[df_expl['iberliferisk'] > 1.0]),
    'IBER > 1.5' : perfil_riesgo(df_expl[df_expl['iberliferisk'] > 1.5]),
})
print(resumen)

                  IBER > 1  IBER > 1.5
N pacientes             34          17
HTA crónica             16           9
IMC ≥ 30                25          15
Colesterol ≥ 200        14           8
HDL < 50                20          13
TAS ≥ 140               11           9
Tabaco (Si/Ex)          28          12
Diabetes                 4           4


In [3]:
# Detalle paciente a paciente (corte > 1) para inspección clínica
cols_detalle = ['id', 'iberliferisk', 'hta_cronica', 'imc_actual',
                'colesterol_total', 'hdl', 'ta_sistolica', 'fuma_post',
                'diabetes_mellitus_1_2']
detalle = (df_expl[df_expl['iberliferisk'] > 1.0][cols_detalle]
           .sort_values('iberliferisk', ascending=False)
           .reset_index(drop=True))
detalle['id'] = detalle['id'].astype(int)
detalle

,id,iberliferisk,hta_cronica,imc_actual,colesterol_total,hdl,ta_sistolica,fuma_post,diabetes_mellitus_1_2
0,333,18.56,1.0,44.99,198.53,35.60,187.0,No,1.0
1,355,5.32,1.0,40.92,146.29,36.77,179.0,No,0.0
2,427,4.84,0.0,30.16,271.29,52.25,133.0,Si,0.0
3,117,4.44,0.0,34.34,244.20,44.51,123.0,Si,1.0
4,136,3.11,1.0,29.89,182.28,56.12,141.0,Ex_Fumadora,1.0
5,320,3.02,0.0,33.41,281.35,54.18,172.0,No,0.0
6,107,2.94,0.0,41.66,230.65,33.67,122.0,Si,0.0
7,255,2.59,1.0,40.78,193.89,35.60,133.0,Si,0.0
8,319,2.44,0.0,34.36,240.33,39.09,135.0,Si,0.0
9,299,2.43,1.0,35.36,178.41,52.63,142.0,Si,0.0


## 2. Flag de riesgo CV en `datos.csv`

Se marca **1** si la paciente tiene **alguno** de los 5 factores de Framingham alterado, **0** si no tiene ninguno.

Notas sobre datos faltantes:
- `fuma_post` y `diabetes_mellitus_1_2` están completos (459/459).
- `colesterol_total` y `hdl` tienen ~31% de missing; `ta_sistolica` casi completo.
- Un missing en un factor **no** marca riesgo por ese factor, pero la paciente puede marcarse por otro.
- Ninguna paciente tiene los 5 factores en blanco a la vez, así que todo `0` significa "sin factor registrado".

In [4]:
# Un flag booleano por factor
factores = pd.DataFrame({
    'colesterol_alto' : df['colesterol_total'] >= CT_ALTO,
    'hdl_bajo'        : df['hdl'] < HDL_MIN,
    'tas_alta'        : df['ta_sistolica'] >= TAS_ALTA,
    'fuma'            : df['fuma_post'].isin(['Si', 'Ex_Fumadora']),
    'diabetes'        : df['diabetes_mellitus_1_2'] == 1,
})

# Flag global: 1 si tiene algún factor, 0 si ninguno
df['riesgo_cv_factor'] = factores.any(axis=1).astype(int)

print('Pacientes con cada factor:')
print(factores.sum().to_string())
print()
print('Flag riesgo_cv_factor:')
print(df['riesgo_cv_factor'].value_counts().rename({1: '1 (con riesgo)', 0: '0 (sin riesgo)'}).to_string())
print(f'\nTotal: {len(df)} pacientes')

Pacientes con cada factor:
colesterol_alto     93
hdl_bajo            84
tas_alta            27
fuma               179
diabetes            11

Flag riesgo_cv_factor:
riesgo_cv_factor
1 (con riesgo)    296
0 (sin riesgo)    163

Total: 459 pacientes


In [5]:
# Guardado.
# Por seguridad se escribe en un fichero NUEVO para no sobrescribir el dataset base.
# Si se prefiere añadir la columna directamente a datos.csv, cambiar la ruta a 'datos.csv'.
df['id'] = df['id'].astype(int)
df.to_csv('datos_riesgo_cv.csv', index=False)
print('Guardado: datos_riesgo_cv.csv  (columna nueva: riesgo_cv_factor)')

Guardado: datos_riesgo_cv.csv  (columna nueva: riesgo_cv_factor)
